# CAMELS-Style Scorecard — Greek Systemic Banks (2022–2024)

A simple 1–5 scoring of public KPIs, in the spirit of the supervisory CAMELS framework.
**1 = strong, 5 = weak.** The cut-off points are my own, set around EBA Risk Dashboard averages for EU banks; they are not an official supervisory method.

| Dimension | Proxy metric | Direction |
|-----------|-------------|----------|
| **C** Capital | CET1 ratio (as reported) | Higher is better |
| **A** Asset quality | NPE ratio (as reported) | Lower is better |
| **M** Management | Cost-to-income ratio | Lower is better |
| **E** Earnings | ROE on average equity | Higher is better |
| **L** Liquidity | Loan-to-deposit ratio | Lower is better |

The **S** (sensitivity to market risk) dimension needs interest rate risk and trading book data that the annual reports give only in different formats, so it is left out.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
YEARS = [2022, 2023, 2024]

DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')
kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
print(f'Loaded {len(kpis)} rows. Columns: {list(kpis.columns)}')

Loaded 12 rows. Columns: ['bank', 'year', 'nii', 'fees', 'operating_income', 'operating_expenses', 'net_profit', 'pbt', 'impairment', 'loans', 'deposits', 'equity', 'at1', 'intangibles', 'tangible_equity', 'total_assets', 'roe', 'rote', 'roa', 'nim', 'cost_of_risk', 'cost_to_income', 'loan_to_deposit', 'fee_share', 'equity_to_assets', 'nii_yoy', 'profit_yoy', 'assets_yoy', 'opincome_yoy', 'cet1', 'total_capital_ratio', 'npe_ratio', 'dtc', 'rwa', 'dtc_to_cet1', 'avg_equity', 'avg_tangible_equity', 'avg_total_assets', 'avg_loans']


In [2]:
# ── CAMELS metrics (all computed in kpis_final.csv) ───────────────────────────
df = kpis.copy()

camels_metrics = {
    'C — Capital':       ('cet1',           'higher'),
    'A — Asset Quality': ('npe_ratio',      'lower'),
    'M — Management':    ('cost_to_income', 'lower'),
    'E — Earnings':      ('roe',            'higher'),
    'L — Liquidity':     ('loan_to_deposit','lower'),
}

print('CAMELS metrics:')
print(df[['bank','year','cet1','npe_ratio','cost_of_risk','cost_to_income','roe','loan_to_deposit']].round(2).to_string(index=False))

CAMELS metrics:
        bank  year  cet1  npe_ratio  cost_of_risk  cost_to_income  roe  loan_to_deposit
  Alpha Bank  2022 13.20        7.8          1.26            48.9  6.0             76.3
  Alpha Bank  2023 14.40        6.0          1.02            38.3  9.1             74.6
  Alpha Bank  2024 16.30        3.8          0.96            39.0  8.5             76.5
    Eurobank  2022 16.00        5.2          0.68            28.2 22.1             72.8
    Eurobank  2023 16.90        3.5          0.99            31.4 15.7             72.3
    Eurobank  2024 16.80        2.9          0.66            32.9 17.2             64.8
         NBG  2022 16.60        5.2          0.66            36.3 18.4             64.4
         NBG  2023 17.80        3.7          0.77            33.6 15.7             59.9
         NBG  2024 18.30        2.6          0.51            36.5 14.4             62.7
Piraeus Bank  2022 13.04        6.8          1.28            34.4 15.4             64.1
Piraeus Bank  20

In [3]:
# ── Scoring thresholds (European banking supervisory calibration) ─────────────
#
# My own cut-off points, set around EBA Risk Dashboard averages for EU banks
# (EU average CET1 c.16%, NPL ratio c.2%, cost-to-income c.53%, RoE c.10%).

def score_metric(value, thresholds, direction):
    """
    Return 1 (best) to 5 (worst).
    thresholds: list of 4 cut-points dividing the [1,2,3,4,5] regions.
    direction:  'higher' (higher = better) or 'lower' (lower = better).
    """
    if pd.isna(value):
        return np.nan
    t = sorted(thresholds)
    if direction == 'higher':
        if   value >= t[3]: return 1
        elif value >= t[2]: return 2
        elif value >= t[1]: return 3
        elif value >= t[0]: return 4
        else:               return 5
    else:  # lower is better
        if   value <= t[0]: return 1
        elif value <= t[1]: return 2
        elif value <= t[2]: return 3
        elif value <= t[3]: return 4
        else:               return 5

THRESHOLDS = {
    'cet1':           ([11.0, 13.0, 15.0, 18.0], 'higher'),  # %
    'npe_ratio':      ([2.0, 3.0, 5.0, 8.0],     'lower'),   # %
    'cost_to_income': ([30.0, 37.0, 45.0, 55.0], 'lower'),   # %
    'roe':            ([5.0,  8.0,  12.0, 15.0],  'higher'),  # %
    'loan_to_deposit':([60.0, 65.0, 75.0, 85.0], 'lower'),   # %
}

score_cols = {}
for dim, (metric, _) in camels_metrics.items():
    thresholds, direction = THRESHOLDS[metric]
    col = f'score_{metric}'
    df[col] = df[metric].apply(lambda v: score_metric(v, thresholds, direction))
    score_cols[dim] = col

# Composite CAMELS score (simple average)
score_col_list = list(score_cols.values())
df['camels_composite'] = df[score_col_list].mean(axis=1).round(1)

print('Scores computed:')
display_df = df[['bank','year'] + score_col_list + ['camels_composite']].copy()
print(display_df.to_string(index=False))

Scores computed:
        bank  year  score_cet1  score_npe_ratio  score_cost_to_income  score_roe  score_loan_to_deposit  camels_composite
  Alpha Bank  2022           3                4                     4          4                      4               3.8
  Alpha Bank  2023           3                4                     3          3                      3               3.2
  Alpha Bank  2024           2                3                     3          3                      4               3.0
    Eurobank  2022           2                4                     1          1                      3               2.2
    Eurobank  2023           2                3                     2          1                      3               2.2
    Eurobank  2024           2                2                     2          1                      2               1.8
         NBG  2022           2                4                     2          1                      2               2.2
       

In [4]:
# ── Chart 1: CAMELS Heatmap — 3 panels (one per year) ────────────────────────
# Each cell: colour = score (1=green, 5=red), annotation = raw metric value

colorscale = [
    [0.0,  '#1a7a4a'],  # 1 — Strong (dark green)
    [0.25, '#5cb85c'],  # 2 — Good (light green)
    [0.5,  '#f0ad4e'],  # 3 — Satisfactory (amber)
    [0.75, '#d9534f'],  # 4 — Marginal (red)
    [1.0,  '#7b1a1a'],  # 5 — Unsatisfactory (dark red)
]

dim_short = ['C (CET1)', 'A (NPE)', 'M (C/I)', 'E (ROE)', 'L (LtD)']
metric_fmts = {
    'cet1':           '{:.1f}%',
    'npe_ratio':      '{:.1f}%',
    'cost_to_income': '{:.1f}%',
    'roe':            '{:.1f}%',
    'loan_to_deposit':'{:.1f}%',
}

fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'<b>{y}</b>' for y in YEARS],
    horizontal_spacing=0.06,
)

metric_order = ['cet1','npe_ratio','cost_to_income','roe','loan_to_deposit']

for ci, year in enumerate(YEARS):
    yr_df = df[df['year'] == year].set_index('bank').reindex(BANKS)

    z_matrix = np.array([[yr_df.loc[b, f'score_{m}'] for m in metric_order] for b in BANKS], dtype=float)
    text_matrix = [
        [metric_fmts[m].format(yr_df.loc[b, m]) + f'<br><b>{int(yr_df.loc[b, "score_" + m])}</b>'
         for m in metric_order]
        for b in BANKS
    ]

    fig1.add_trace(
        go.Heatmap(
            z=z_matrix,
            x=dim_short,
            y=BANKS,
            text=text_matrix,
            texttemplate='%{text}',
            colorscale=colorscale,
            zmin=1, zmax=5,
            showscale=(ci == 2),
            colorbar=dict(title='Score', tickvals=[1,2,3,4,5],
                          ticktext=['1 Strong','2 Good','3 OK','4 Marginal','5 Weak'],
                          len=0.6),
            hoverongaps=False,
        ),
        row=1, col=ci + 1,
    )

fig1.update_layout(
    title_text='<b>CAMELS Scorecard</b> | Greek Systemic Banks 2022–2024 | 1 = Strong, 5 = Weak',
    title_font_size=16,
    template='plotly_dark',
    height=400,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
    font=dict(size=12),
)
fig1.show()

In [5]:
# ── Chart 2: Composite CAMELS Score Trend ─────────────────────────────────────
fig2 = go.Figure()
for bank in BANKS:
    bdf = df[df['bank'] == bank].sort_values('year')
    fig2.add_trace(go.Scatter(
        x=bdf['year'],
        y=bdf['camels_composite'],
        name=bank,
        mode='lines+markers+text',
        line=dict(color=COLORS[bank], width=2.5),
        marker=dict(size=10),
        text=bdf['camels_composite'].astype(str),
        textposition='top center',
    ))

fig2.update_layout(
    title_text='<b>Composite CAMELS Score</b> | Simple average of 5 dimensions | Lower = Stronger',
    xaxis=dict(tickvals=YEARS, ticktext=[str(y) for y in YEARS]),
    yaxis=dict(title='Composite Score (1=Strong, 5=Weak)', range=[0.5, 5.5], autorange='reversed'),
    template='plotly_dark',
    height=380,
    legend=dict(orientation='h', y=-0.18, x=0.5, xanchor='center'),
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig2.show()

In [6]:
# ── Chart 3: 2024 Ranked Summary Table ───────────────────────────────────────
df24 = df[df['year'] == 2024].copy().sort_values('camels_composite')

print('\n── 2024 CAMELS Rankings (sorted by composite score, lower = stronger) ──')
rank_df = df24[['bank'] + [f'score_{m}' for m in metric_order] + ['camels_composite']].copy()
rank_df.columns = ['Bank', 'C (Capital)', 'A (Asset)', 'M (Mgmt)', 'E (Earn)', 'L (Liq)', 'Composite']
rank_df = rank_df.reset_index(drop=True)
rank_df.index = rank_df.index + 1
rank_df.index.name = 'Rank'
print(rank_df.to_string())

print('\n── Threshold Reference ──')
thresh_info = [
    ('C (CET1)',        '> 18%',    '15-18%',  '13-15%',   '11-13%',   '< 11%'),
    ('A (NPE ratio)',   '< 2%',     '2-3%',    '3-5%',     '5-8%',     '> 8%'),
    ('M (Cost/Income)', '< 30%',    '30-37%',  '37-45%',   '45-55%',   '> 55%'),
    ('E (ROE)',         '> 15%',    '12-15%',  '8-12%',    '5-8%',     '< 5%'),
    ('L (Loan/Dep.)',   '< 60%',    '60-65%',  '65-75%',   '75-85%',   '> 85%'),
]
header = f'  {"Dimension":<18} {"Score 1":<12} {"Score 2":<12} {"Score 3":<12} {"Score 4":<12} {"Score 5"}'
print(header)
print('  ' + '-' * 80)
for row in thresh_info:
    print(f'  {row[0]:<18} {row[1]:<12} {row[2]:<12} {row[3]:<12} {row[4]:<12} {row[5]}')


── 2024 CAMELS Rankings (sorted by composite score, lower = stronger) ──
              Bank  C (Capital)  A (Asset)  M (Mgmt)  E (Earn)  L (Liq)  Composite
Rank                                                                              
1         Eurobank            2          2         2         1        2        1.8
2              NBG            1          2         2         2        2        1.8
3     Piraeus Bank            3          2         2         2        2        2.2
4       Alpha Bank            2          3         3         3        4        3.0

── Threshold Reference ──
  Dimension          Score 1      Score 2      Score 3      Score 4      Score 5
  --------------------------------------------------------------------------------
  C (CET1)           > 18%        15-18%       13-15%       11-13%       < 11%
  A (NPE ratio)      < 2%         2-3%         3-5%         5-8%         > 8%
  M (Cost/Income)    < 30%        30-37%       37-45%       45-55%       > 55%
 

In [7]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(df) == 12, 'Expected 12 bank-year rows.'
assert df['npe_ratio'].notna().all(), 'Missing NPE ratios.'
assert df['camels_composite'].between(1, 5).all(), 'Composite score out of [1,5] range.'
for col in score_col_list:
    assert df[col].between(1, 5).all(), f'{col} has values outside [1,5].'

nbg = df[(df['bank'] == 'NBG') & (df['year'] == 2024)].iloc[0]
assert nbg['score_cet1'] <= 2, f'NBG 2024 CET1 {nbg["cet1"]}% should score 1-2'

print('All checks passed.')
print(f'   2024 sector average composite score: {df[df["year"]==2024]["camels_composite"].mean():.2f}')
print(f'   Strongest bank (2024): {df24.iloc[0]["bank"]} (composite {df24.iloc[0]["camels_composite"]})')
print(f'   Weakest bank  (2024): {df24.iloc[-1]["bank"]} (composite {df24.iloc[-1]["camels_composite"]})')

All checks passed.
   2024 sector average composite score: 2.20
   Strongest bank (2024): Eurobank (composite 1.8)
   Weakest bank  (2024): Alpha Bank (composite 3.0)
